# Ropedia Academy — B2 · Multi-view geometry & triangulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B2.ipynb)

> **Two cameras → project a point → recover it by DLT triangulation, reporting the error; the epipolar 1D-search insight is called out.**
>
> 两个相机 → 投影一个点 → 用 DLT 三角测量还原它并报告误差；点出对极约束把匹配从 2D 降为 1D。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B2

In [ ]:
import numpy as np

# Two calibrated views observe one 3D point; recover it by triangulation.
K = np.array([[800,0,320],[0,800,240],[0,0,1.]])
cam = lambda R, t: K @ np.hstack([R, t.reshape(3,1)])   # 3x4 projection matrix
P1 = cam(np.eye(3), np.zeros(3))
a = 0.3; R2 = np.array([[np.cos(a),0,np.sin(a)],[0,1,0],[-np.sin(a),0,np.cos(a)]])
P2 = cam(R2, np.array([-1., 0, 0]))

X_true = np.array([0.5, -0.2, 4.0, 1.0])           # homogeneous 3D point
proj = lambda P, X: (P @ X)[:2] / (P @ X)[2]
x1, x2 = proj(P1, X_true), proj(P2, X_true)         # its two images (correspondence)

def triangulate(P1, P2, x1, x2):                    # DLT: solve the ray intersection (LSQ)
    A = np.stack([x1[0]*P1[2]-P1[0], x1[1]*P1[2]-P1[1],
                  x2[0]*P2[2]-P2[0], x2[1]*P2[2]-P2[1]])
    _, _, Vt = np.linalg.svd(A); X = Vt[-1]; return X / X[3]

X_est = triangulate(P1, P2, x1, x2)
print("recovered:", X_est[:3].round(3), "| true:", X_true[:3])
print("triangulation error:", round(np.linalg.norm(X_est - X_true), 5),
      " (epipolar constraint reduces matching from 2D to a 1D line search)")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B2
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks